In [0]:
-- ============================================================
-- Department variance analysis: project vs department average
-- ============================================================

WITH dept_avg AS (
    SELECT
        d.department_name,
        ROUND(AVG(
            p.completed_tasks / NULLIF(p.total_tasks, 0) * 100
        ), 1) AS dept_avg_pct
    FROM healthcare_analytics.projects p
    JOIN healthcare_analytics.departments d
        ON p.department_id = d.department_id
    GROUP BY d.department_name
),

project_detail AS (
    SELECT
        p.project_id,
        p.project_name,
        d.department_name,
        ROUND(
            p.completed_tasks / NULLIF(p.total_tasks, 0) * 100, 1
        ) AS completion_pct
    FROM healthcare_analytics.projects p
    JOIN healthcare_analytics.departments d
        ON p.department_id = d.department_id
)

SELECT
    pd.project_id,
    pd.project_name,
    pd.department_name,
    pd.completion_pct,
    da.dept_avg_pct,
    ROUND(pd.completion_pct - da.dept_avg_pct, 1) AS variance_vs_avg,
    CASE
        WHEN pd.completion_pct >= da.dept_avg_pct + 10 THEN 'Outperforming'
        WHEN pd.completion_pct <= da.dept_avg_pct - 10 THEN 'Underperforming'
        ELSE 'Within range'
    END AS performance_vs_dept
FROM project_detail pd
JOIN dept_avg da
    ON pd.department_name = da.department_name
ORDER BY pd.department_name, pd.completion_pct DESC;

project_id,project_name,department_name,completion_pct,dept_avg_pct,variance_vs_avg,performance_vs_dept
101,EHR Migration,Clinical Operations,90.0,65.0,25.0,Outperforming
102,Patient Intake Redesign,Clinical Operations,40.0,65.0,-25.0,Underperforming
107,Data Quality Framework,Data & Analytics,84.0,57.0,27.0,Outperforming
108,Report Automation,Data & Analytics,30.0,57.0,-27.0,Underperforming
103,BI Dashboard Rollout,Health Informatics,96.0,78.0,18.0,Outperforming
104,Claims Data Cleanup,Health Informatics,60.0,78.0,-18.0,Underperforming
105,Audit Prep Q1,Quality Assurance,100.0,85.0,15.0,Outperforming
106,Compliance Review,Quality Assurance,70.0,85.0,-15.0,Underperforming


In [0]:
-- ============================================================
-- Window Functions: Rank projects within each department
-- ============================================================

WITH project_metrics AS (
    SELECT
        p.project_id,
        p.project_name,
        d.department_name,
        p.completed_tasks,
        p.total_tasks,
        ROUND(
            p.completed_tasks / NULLIF(p.total_tasks, 0) * 100, 1
        ) AS completion_pct
    FROM healthcare_analytics.projects p
    JOIN healthcare_analytics.departments d
        ON p.department_id = d.department_id
)

SELECT
    project_id,
    project_name,
    department_name,
    completion_pct,

    -- Rank within department
    RANK() OVER (
        PARTITION BY department_name
        ORDER BY completion_pct DESC
    ) AS rank_in_dept,

    -- Running total of completion % within department
    ROUND(SUM(completion_pct) OVER (
        PARTITION BY department_name
        ORDER BY completion_pct DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 1) AS running_total_pct,

    -- Previous project completion rate
    LAG(completion_pct, 1) OVER (
        PARTITION BY department_name
        ORDER BY project_id
    ) AS prev_project_pct,

    -- Performance tier
    CASE
        WHEN completion_pct >= 90 THEN 'High'
        WHEN completion_pct >= 60 THEN 'On Track'
        ELSE 'At Risk'
    END AS performance_tier

FROM project_metrics
ORDER BY department_name, rank_in_dept;

project_id,project_name,department_name,completion_pct,rank_in_dept,running_total_pct,prev_project_pct,performance_tier
101,EHR Migration,Clinical Operations,90.0,1,90.0,null,High
102,Patient Intake Redesign,Clinical Operations,40.0,2,130.0,90.0,At Risk
107,Data Quality Framework,Data & Analytics,84.0,1,84.0,null,On Track
108,Report Automation,Data & Analytics,30.0,2,114.0,84.0,At Risk
103,BI Dashboard Rollout,Health Informatics,96.0,1,96.0,null,High
104,Claims Data Cleanup,Health Informatics,60.0,2,156.0,96.0,On Track
105,Audit Prep Q1,Quality Assurance,100.0,1,100.0,null,High
106,Compliance Review,Quality Assurance,70.0,2,170.0,100.0,On Track


In [0]:
-- Insert department data
INSERT INTO healthcare_analytics.departments VALUES
    (1, 'Clinical Operations'),
    (2, 'Health Informatics'),
    (3, 'Quality Assurance'),
    (4, 'Data & Analytics');

-- Insert project data
INSERT INTO healthcare_analytics.projects VALUES
    (101, 'EHR Migration',           1, 45, 50, '2024-03-01'),
    (102, 'Patient Intake Redesign', 1, 20, 50, '2024-04-15'),
    (103, 'BI Dashboard Rollout',    2, 48, 50, '2024-02-28'),
    (104, 'Claims Data Cleanup',     2, 30, 50, '2024-05-01'),
    (105, 'Audit Prep Q1',           3, 50, 50, '2024-01-31'),
    (106, 'Compliance Review',       3, 35, 50, '2024-03-15'),
    (107, 'Data Quality Framework',  4, 42, 50, '2024-04-01'),
    (108, 'Report Automation',       4, 15, 50, '2024-06-01');

num_affected_rows,num_inserted_rows
8,8


In [0]:
-- Create departments table
CREATE TABLE IF NOT EXISTS healthcare_analytics.departments (
    department_id   INT,
    department_name STRING
);

-- Create projects table
CREATE TABLE IF NOT EXISTS healthcare_analytics.projects (
    project_id       INT,
    project_name     STRING,
    department_id    INT,
    completed_tasks  INT,
    total_tasks      INT,
    due_date         DATE
);

In [0]:
-- ============================================================
-- Notebook 01: SQL Window Functions in Databricks
-- Author: Alejandra Kheng
-- Description: Recreates project completion analytics using
--              Databricks SQL. Demonstrates CTEs, window
--              functions, and aggregations on Delta tables.
-- ============================================================

-- Create database to organize our tables
CREATE DATABASE IF NOT EXISTS healthcare_analytics;
USE healthcare_analytics;
